**Create Raw Data Set for Streaming**

In [0]:
from pyspark.sql import Row
from datetime import datetime, timedelta
import random

# Simulate e-commerce events (similar shape to what you'll stream later)
event_types = ["page_view", "add_to_cart", "purchase"]
products = ["laptop", "phone", "headphones", "monitor", "keyboard"]

data = []
base_time = datetime(2026, 8, 1, 9, 0, 0)
for i in range(2000):
    data.append(Row(
        user_id=f"user_{random.randint(1, 200)}",
        event_type=random.choice(event_types),
        product=random.choice(products),
        price=round(random.uniform(10, 1500), 2),
        event_time=base_time + timedelta(seconds=random.randint(0, 3600*5))
    ))

df = spark.createDataFrame(data)
df.display()

user_id,event_type,product,price,event_time
user_5,purchase,laptop,112.33,2026-08-01T10:23:13.000Z
user_158,add_to_cart,phone,1104.34,2026-08-01T12:10:44.000Z
user_53,page_view,monitor,458.83,2026-08-01T12:05:49.000Z
user_23,page_view,headphones,788.9,2026-08-01T12:10:39.000Z
user_45,page_view,headphones,1344.08,2026-08-01T10:18:59.000Z
user_33,purchase,monitor,477.78,2026-08-01T10:31:31.000Z
user_2,add_to_cart,headphones,207.78,2026-08-01T12:04:34.000Z
user_125,purchase,phone,1360.85,2026-08-01T12:54:24.000Z
user_49,page_view,keyboard,474.44,2026-08-01T10:34:43.000Z
user_155,add_to_cart,keyboard,499.3,2026-08-01T12:49:53.000Z


**Print Schema**

In [0]:
df.printSchema()
df.count()
df.groupBy("event_type").count().display()

root
 |-- user_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product: string (nullable = true)
 |-- price: double (nullable = true)
 |-- event_time: timestamp (nullable = true)



event_type,count
page_view,652
add_to_cart,709
purchase,639


**Filter & Select**

In [0]:
purchases = df.filter(df.event_type == "purchase").select("user_id", "product", "price", "event_time")
purchases.display()

user_id,product,price,event_time
user_5,laptop,112.33,2026-08-01T10:23:13.000Z
user_33,monitor,477.78,2026-08-01T10:31:31.000Z
user_125,phone,1360.85,2026-08-01T12:54:24.000Z
user_73,keyboard,488.66,2026-08-01T12:16:33.000Z
user_48,phone,234.22,2026-08-01T10:33:25.000Z
user_63,phone,96.18,2026-08-01T11:45:18.000Z
user_8,laptop,287.23,2026-08-01T13:16:58.000Z
user_128,headphones,1355.0,2026-08-01T12:24:17.000Z
user_67,headphones,781.44,2026-08-01T13:11:29.000Z
user_80,keyboard,573.86,2026-08-01T11:15:22.000Z


**GroupBy & Aggregation**

In [0]:
from pyspark.sql.functions import sum as _sum, count, avg

revenue_by_product = (
    df.filter(df.event_type == "purchase")
      .groupBy("product")
      .agg(
          _sum("price").alias("total_revenue"),
          count("*").alias("num_purchases"),
          avg("price").alias("avg_price")
      )
      .orderBy("total_revenue", ascending=False)
)
revenue_by_product.display()

product,total_revenue,num_purchases,avg_price
headphones,101995.29999999999,134,761.1589552238805
phone,100485.41,124,810.3662096774194
laptop,99557.13000000003,131,759.9780916030537
monitor,93181.33,129,722.3358914728682
keyboard,87945.31,121,726.8207438016528


****

**With Coulmn**

In [0]:
from pyspark.sql.functions import when, col

df_flagged = df.withColumn(
    "high_value",
    when(col("price") > 800, True).otherwise(False)
)
df_flagged.display()

user_id,event_type,product,price,event_time,high_value
user_5,purchase,laptop,112.33,2026-08-01T10:23:13.000Z,false
user_158,add_to_cart,phone,1104.34,2026-08-01T12:10:44.000Z,true
user_53,page_view,monitor,458.83,2026-08-01T12:05:49.000Z,false
user_23,page_view,headphones,788.9,2026-08-01T12:10:39.000Z,false
user_45,page_view,headphones,1344.08,2026-08-01T10:18:59.000Z,true
user_33,purchase,monitor,477.78,2026-08-01T10:31:31.000Z,false
user_2,add_to_cart,headphones,207.78,2026-08-01T12:04:34.000Z,false
user_125,purchase,phone,1360.85,2026-08-01T12:54:24.000Z,true
user_49,page_view,keyboard,474.44,2026-08-01T10:34:43.000Z,false
user_155,add_to_cart,keyboard,499.3,2026-08-01T12:49:53.000Z,false


**Delta table "Bronze layer**

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("bronze_events")

spark.sql("SELECT event_type, COUNT(*) FROM bronze_events GROUP BY event_type").display()

event_type,COUNT(*)
page_view,652
add_to_cart,709
purchase,639


**Left-join need this for Silver-layer enrichment later**

In [0]:
product_catalog = spark.createDataFrame([
    ("laptop", "Electronics"), ("phone", "Electronics"),
    ("headphones", "Accessories"), ("monitor", "Electronics"),
    ("keyboard", "Accessories")
], ["product", "category"])

enriched = df.join(product_catalog, on="product", how="left")
enriched.groupBy("category").count().display()

category,count
Electronics,1203
Accessories,797


**Top 5 users by total purchase amount**

In [0]:
from pyspark.sql.functions import sum as _sum

top_users = (
    df.filter(df.event_type == "purchase")
      .groupBy("user_id")
      .agg(
          _sum("price").alias("total_purchase_amount")
      )
      .orderBy("total_purchase_amount", ascending=False)
      .limit(5)
)
top_users.display()

user_id,total_purchase_amount
user_117,7379.11
user_89,6649.849999999999
user_69,6499.110000000001
user_107,5972.27
user_184,5965.950000000001


**Anti Join**

In [0]:
add_to_cart_users = df.filter(df.event_type == "add_to_cart").select("user_id").distinct()
purchase_users = df.filter(df.event_type == "purchase").select("user_id").distinct()

exclusive_add_to_cart = add_to_cart_users.join(purchase_users, on="user_id", how="left_anti")
exclusive_add_to_cart_count = exclusive_add_to_cart.count()
exclusive_add_to_cart_count

10

**Event Time**

In [0]:
from pyspark.sql.functions import window

purchases_windowed = (
    df.filter(df.event_type == "purchase")
      .groupBy(window("event_time", "15 minutes"))
      .count()
      .orderBy("window")
)

purchases_windowed.display()

window,count
"List(2026-08-01T09:00:00.000Z, 2026-08-01T09:15:00.000Z)",34
"List(2026-08-01T09:15:00.000Z, 2026-08-01T09:30:00.000Z)",27
"List(2026-08-01T09:30:00.000Z, 2026-08-01T09:45:00.000Z)",30
"List(2026-08-01T09:45:00.000Z, 2026-08-01T10:00:00.000Z)",30
"List(2026-08-01T10:00:00.000Z, 2026-08-01T10:15:00.000Z)",27
"List(2026-08-01T10:15:00.000Z, 2026-08-01T10:30:00.000Z)",36
"List(2026-08-01T10:30:00.000Z, 2026-08-01T10:45:00.000Z)",31
"List(2026-08-01T10:45:00.000Z, 2026-08-01T11:00:00.000Z)",35
"List(2026-08-01T11:00:00.000Z, 2026-08-01T11:15:00.000Z)",33
"List(2026-08-01T11:15:00.000Z, 2026-08-01T11:30:00.000Z)",30
